In [1]:
!pip install "awswrangler[redshift]" --no-build-isolation

In [2]:
import pandas as pd
import random

import boto3
import io

import awswrangler as wr

from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

In [3]:
# Semilla para reproducibilidad (se puede cambiar)
random.seed(42)

In [4]:
# FECHA_REC = "2026-03-25"
manana_lima = datetime.now(ZoneInfo("America/Lima")) + timedelta(days=1)
FECHA_REC = manana_lima.strftime("%Y-%m-%d")
FECHA_REC

'2026-03-31'

In [5]:
# 1. Configurar cliente y separar Bucket de la Ruta del archivo
s3 = boto3.client('s3')
bucket_name = "aje-prd-analytics-artifacts-s3"
file_key = "pedido_sugerido/data-v1/ecuador/ventas_ecuador_tomorrow.csv"

# 2. Descargar el objeto del S3 a la memoria RAM
response = s3.get_object(Bucket=bucket_name, Key=file_key)

# 3. Leerlo con Pandas sin pasar por s3fs
cl = pd.read_csv(io.BytesIO(response['Body'].read()))

In [6]:
cl[["id_sucursal","desc_sucursal"]].drop_duplicates()

,id_sucursal,desc_sucursal
0,EC|0090|17,CEDIS GUAYAQUIL SUR
1,EC|0090|04,CEDIS QUITO NORTE
2,EC|0090|07,CEDIS MANTA
3,EC|0090|11,CEDIS QUITO SUR
5,EC|0090|06,CEDIS MACHALA
7,EC|0090|09,CEDIS SANTO DOMINGO
11,EC|0090|08,CEDIS AMBATO
27,EC|0090|03,CEDIS GUAYAQUIL NORTE


In [7]:
RUTAS_ADICIONALES = [1903,1703]
RUTAS_ADICIONALES

[1903, 1703]

In [8]:
SUCURSALES = ["EC|0090|07"]
SUCURSALES

['EC|0090|07']

In [9]:
cl = cl[(cl.id_sucursal.isin(SUCURSALES)) | (cl.cod_ruta.isin(RUTAS_ADICIONALES))][["cod_compania","cod_sucursal","cod_modulo","cod_cliente"]].drop_duplicates().reset_index(drop=True)
# cl = cl[cl.cod_ruta.isin(RUTAS_ADICIONALES)][["cod_compania","cod_sucursal","cod_modulo","cod_cliente"]].drop_duplicates().reset_index(drop=True)

In [10]:
# Modulos en total
cl.cod_modulo.nunique()

24

In [11]:
print(sorted(cl.cod_modulo.unique()))

[64, 96, 332, 336, 337, 338, 341, 355, 357, 358, 359, 361, 362, 365, 366, 368, 369, 370, 371, 375, 385, 386, 90704, 90707]


In [12]:
cl.shape

(1788, 4)

In [13]:
df_unicos = cl[['cod_compania', 'cod_cliente']].drop_duplicates()

In [14]:
cl["Pais"] = "EC"
cl["Cajas"] = 1
cl["Unidades"] = 0
cl["Fecha"] = FECHA_REC

In [15]:
productos = [508462, 524187, 524188, 524189, 515517, 515519, 500010, 599653, 500147]

In [16]:
df_prod = pd.DataFrame({'Producto': productos})

In [17]:
# 3. Producto cartesiano entre combinaciones únicas y productos
df_combinado = df_unicos.merge(df_prod, how='cross')

# 4. Cruzar de vuelta con el DataFrame original para mantener el resto de columnas
df_final = df_combinado.merge(cl, on=['cod_compania', 'cod_cliente'])

In [18]:
df_final.columns = ['Compania', 'Cliente', 'Producto', 'Sucursal', 'Modulo',
       'Pais', 'Cajas', 'Unidades', 'Fecha']
df_final = df_final[['Pais','Compania','Sucursal','Cliente','Modulo','Producto','Cajas','Unidades','Fecha']]

In [19]:
# Crear el secuencial por grupo (0, 1, ...) y sumarle 1 para que empiece en 1
secuencia = df_final.groupby(['Compania', 'Cliente']).cumcount() + 1

# Crear la columna tipoRecomendacion concatenando 'PE' con el número
df_final['tipoRecomendacion'] = 'PE' + secuencia.astype(str)

In [20]:
df_final["ultFecha"] = ''
df_final["Destacar"] = "true"

In [21]:
df_final["Cliente"] = df_final["Cliente"].astype(int)
df_final["Producto"] = df_final["Producto"].astype(int)
df_final["Unidades"] = df_final["Unidades"].astype(int)
df_final["Cajas"] = df_final["Cajas"].astype(int)

df_final["Compania"] = df_final["Compania"].astype(str).str.zfill(4)
df_final["Sucursal"] = df_final["Sucursal"].astype(str).str.zfill(2)

In [22]:
df_final["Cliente"].nunique()

1788

In [23]:
df_final

,Pais,Compania,Sucursal,Cliente,Modulo,Producto,Cajas,Unidades,Fecha,tipoRecomendacion,ultFecha,Destacar
0,EC,0090,07,482295,368,508462,1,0,2026-03-31,PE1,,true
1,EC,0090,07,482295,368,524187,1,0,2026-03-31,PE2,,true
2,EC,0090,07,482295,368,524188,1,0,2026-03-31,PE3,,true
3,EC,0090,07,482295,368,524189,1,0,2026-03-31,PE4,,true
4,EC,0090,07,482295,368,515517,1,0,2026-03-31,PE5,,true
...,...,...,...,...,...,...,...,...,...,...,...,...
16087,EC,0090,07,445105,336,515517,1,0,2026-03-31,PE5,,true
16088,EC,0090,07,445105,336,515519,1,0,2026-03-31,PE6,,true
16089,EC,0090,07,445105,336,500010,1,0,2026-03-31,PE7,,true
16090,EC,0090,07,445105,336,599653,1,0,2026-03-31,PE8,,true


---

---

# Quitamos Recurrente y Sugerido

---

---

In [24]:
FECHA_REC

'2026-03-31'

In [25]:
ps_ec = pd.read_csv(f"s3://aje-analytics-ps-backup/PS_Ecuador/Output/PS_piloto_v1/D_base_pedidos_{FECHA_REC}.csv")
ps_ec = ps_ec[["Compania","Cliente","Producto"]]
pr_ec = pd.read_csv(f"s3://aje-analytics-ps-backup/Pedido_Recurrente/Ecuador/Output/recu_base_pedidos_{FECHA_REC}.csv")
pr_ec = pr_ec[["Compania","Cliente","Producto"]]

/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/fsspec/registry.py:301: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)


In [26]:
ps_ec.Cliente.nunique()

18092

In [27]:
pr_ec.Cliente.nunique()

1634

In [28]:
quitar_temp = pd.concat([pr_ec, ps_ec], ignore_index=True).drop_duplicates()

In [29]:
quitar_temp["Compania"] = quitar_temp["Compania"].astype(str).str.zfill(4)
quitar_temp["id_cliente"] = "EC" + '|' + quitar_temp['Compania'] + '|' + quitar_temp['Cliente'].astype(str)
quitar_temp.rename(columns={"Producto":"cod_articulo_magic"},inplace=True)

In [30]:
quitar_temp

,Compania,Cliente,cod_articulo_magic,id_cliente
0,0090,109330,508514,EC|0090|109330
1,0090,109330,515408,EC|0090|109330
2,0090,109330,514422,EC|0090|109330
3,0090,109330,515904,EC|0090|109330
4,0090,109330,517260,EC|0090|109330
...,...,...,...,...
50971,0090,9996,621870,EC|0090|9996
50972,0090,9996,623423,EC|0090|9996
50973,0090,9999,500010,EC|0090|9999
50974,0090,9999,511920,EC|0090|9999


In [31]:
df_final.shape

(16092, 12)

In [32]:
df_final["id_cliente"] = "EC" + '|' + df_final['Compania'] + '|' + df_final['Cliente'].astype(str)
df_final.rename(columns={"Producto":"cod_articulo_magic"},inplace=True)

In [33]:
# quitamos estrategico y recurrente a nivel cliente
 
# 1. Hacemos un merge por las columnas de coincidencia usando indicator=True
merge_temp = df_final.merge(
    quitar_temp[['id_cliente', 'cod_articulo_magic']], 
    on=['id_cliente', 'cod_articulo_magic'], 
    how='left', 
    indicator=True
)

# 2. Nos quedamos solo con las filas que solo existen en pan_rec (left_only)
rec_sin_pr_ni_ps = merge_temp[merge_temp['_merge'] == 'left_only'].drop(columns=['_merge'])

In [34]:
rec_sin_pr_ni_ps.shape

(11702, 13)

In [35]:
rec_sin_pr_ni_ps.rename(columns={"cod_articulo_magic":"Producto"},inplace=True)
rec_sin_pr_ni_ps.drop(columns=["id_cliente"],inplace=True)

In [36]:
df_final = rec_sin_pr_ni_ps.copy()

---

---

In [37]:
# Nos quedamos con el top 3 por cliente
df_final = df_final.groupby(['Pais','Compania','Sucursal','Cliente']).head(3).reset_index(drop=True)

In [38]:
# Recalculamos su tipoRecomendacion
# Crear el secuencial por grupo (0, 1, ...) y sumarle 1 para que empiece en 1
secuencia = df_final.groupby(['Compania', 'Cliente']).cumcount() + 1

# Crear la columna tipoRecomendacion concatenando 'PE' con el número
df_final['tipoRecomendacion'] = 'PE' + secuencia.astype(str)

In [39]:
df_final.Modulo.unique()

array([  368,   332,   366,   341,   375,   365,   359,   355,   370,
         336,   338,   337,    64,   371, 90704,   357,    96,   358,
         361,   369,   362, 90707,   386,   385])

In [40]:
# Total de clientes sugeridos
df_final.groupby(["Compania"]).Cliente.nunique().sum()

1788

In [41]:
# Total de clientes sugeridos por compania
df_final.groupby(["Compania"]).Cliente.nunique()

Compania
0090    1788
Name: Cliente, dtype: int64

In [42]:
# Total de RECOMENDACIONES por cliente
temp = df_final.groupby(["Pais","Compania","Sucursal","Cliente"]).agg({"Producto":["size","nunique"]}).reset_index()
temp.columns=["Pais","Compania","Sucursal","Cliente","total_productos","productos_unicos"]
temp["Cumple"]= (temp.total_productos ==temp.productos_unicos).astype(int)

In [43]:
temp[temp.Cumple==0]

,Pais,Compania,Sucursal,Cliente,total_productos,productos_unicos,Cumple


In [44]:
temp

,Pais,Compania,Sucursal,Cliente,total_productos,productos_unicos,Cumple
0,EC,0090,03,4273,3,3,1
1,EC,0090,03,4392,3,3,1
2,EC,0090,03,4891,3,3,1
3,EC,0090,03,4901,3,3,1
4,EC,0090,03,4906,3,3,1
...,...,...,...,...,...,...,...
1783,EC,0090,07,487484,3,3,1
1784,EC,0090,07,487534,3,3,1
1785,EC,0090,07,487597,3,3,1
1786,EC,0090,07,487657,3,3,1


In [45]:
df_final

,Pais,Compania,Sucursal,Cliente,Modulo,Producto,Cajas,Unidades,Fecha,tipoRecomendacion,ultFecha,Destacar
0,EC,0090,07,482295,368,508462,1,0,2026-03-31,PE1,,true
1,EC,0090,07,482295,368,524187,1,0,2026-03-31,PE2,,true
2,EC,0090,07,482295,368,524188,1,0,2026-03-31,PE3,,true
3,EC,0090,07,84834,332,508462,1,0,2026-03-31,PE1,,true
4,EC,0090,07,84834,332,524187,1,0,2026-03-31,PE2,,true
...,...,...,...,...,...,...,...,...,...,...,...,...
5359,EC,0090,07,474628,359,524187,1,0,2026-03-31,PE2,,true
5360,EC,0090,07,474628,359,524188,1,0,2026-03-31,PE3,,true
5361,EC,0090,07,445105,336,508462,1,0,2026-03-31,PE1,,true
5362,EC,0090,07,445105,336,524187,1,0,2026-03-31,PE2,,true


In [46]:
# Establecer la conexión con S3
bucket_name = 'aje-analytics-ps-backup'  # nombre de bucket en S3
file_name = f'Pedido_Estrategico/Ecuador/Output/estr_base_pedidos_{FECHA_REC}.csv'  # nombre para el archivo en S3
s3_path = f's3://{bucket_name}/{file_name}'

# Escribir el dataframe en S3 con AWS Data Wrangler
wr.s3.to_csv(df_final, s3_path, index=False)

{'paths': ['s3://aje-analytics-ps-backup/Pedido_Estrategico/Ecuador/Output/estr_base_pedidos_2026-03-31.csv'],
 'partitions_values': {}}

In [47]:
df_final.to_csv(f"Output/EC_PE_base_pedidos_{FECHA_REC}.csv",index=False)